# HeliOS YOLO — Fine-tuning para Detecção de Manchas Solares
**HeliOS | Global Solution 2026.1 — FIAP | 2º Ano TIAO**

---

## Instruções de uso

1. **Runtime**: `Runtime > Change runtime type > T4 GPU > Save`
2. Execute as células **em ordem**, uma por vez
3. Ao final, baixe os arquivos da pasta `/content/yolo_output/`:
   - `helios_yolo.pt` → coloque em `src/ml/yolo/model/`
   - `yolo_metrics.json` → coloque em `src/ml/yolo/model/`
   - `yolo_results.png` → coloque em `docs/`

---

### O que este notebook faz

| Etapa | Descrição |
|-------|-----------|
| 1 | Verifica GPU disponível |
| 2 | Instala `ultralytics` (YOLOv8/v11) |
| 3 | Baixa dataset solar anotado do Roboflow Universe |
| 4 | Prepara estrutura do dataset (YOLO format) |
| 5 | Fine-tuning YOLOv8n (30 épocas, ~10 min na T4) |
| 6 | Avalia métricas (mAP50, Precision, Recall) |
| 7 | Testa inferência em imagem SDO real |
| 8 | Exporta pesos e métricas para download |

## Célula 1 — Verificar GPU

In [ ]:
import subprocess, sys

gpu_info = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if gpu_info.returncode == 0:
    print("✅ GPU disponível:")
    # Mostra apenas nome e memória
    for line in gpu_info.stdout.split("\n"):
        if "MiB" in line or "Tesla" in line or "T4" in line or "A100" in line or "V100" in line:
            print(" ", line)
else:
    print("⚠️  GPU não detectada. Mude para T4: Runtime > Change runtime type > T4 GPU")
    print("   O treinamento em CPU pode levar horas.")

## Célula 2 — Instalar Dependências

In [ ]:
%pip install -q ultralytics roboflow
import ultralytics
print(f"ultralytics: {ultralytics.__version__}")
ultralytics.checks()

## Célula 3 — Baixar Dataset Solar (Roboflow Universe)

Usamos o dataset **"Solar Spots"** do Roboflow Universe — público, ~600 imagens de manchas solares SDO já anotadas no formato YOLO.

> **Nota:** Se o download automático falhar (chave expirada), use o **Plano B** na Célula 3B.

In [ ]:
from roboflow import Roboflow
from pathlib import Path
import os

# Dataset público: "sunspot-detection" no Roboflow Universe
# https://universe.roboflow.com/solar-o8tix/sunspot-detection
# Chave pública de acesso (read-only, sem autenticação necessária)
rf = Roboflow(api_key="YOUR_ROBOFLOW_API_KEY")

# ─── SUBSTITUA pelos valores do dataset que você acessar em ──────────────────
# https://universe.roboflow.com  → buscar "sunspot" ou "solar active region"
# Clicar em "Download Dataset" > "YOLOv8" > "show download code"
# ─────────────────────────────────────────────────────────────────────────────
project  = rf.workspace("solar-o8tix").project("sunspot-detection")
dataset  = project.version(1).download("yolov8")

DATASET_PATH = Path(dataset.location)
print(f"\n✅ Dataset baixado em: {DATASET_PATH}")
print(f"   Conteúdo: {list(DATASET_PATH.iterdir())}")

## Célula 3B — Plano B: Gerar Dataset Sintético

> **Execute esta célula APENAS se a Célula 3 falhar.**  
> Gera ~300 imagens sintéticas de manchas solares com bounding boxes automaticamente anotadas — suficiente para demonstrar o pipeline YOLO academicamente.

In [ ]:
"""
Gerador de dataset sintético solar para YOLO.
Cria imagens de disco solar com manchas simuladas + anotações YOLO (.txt).

Classes:
  0 — quiet_sun      (sem manchas detectáveis)
  1 — active_region  (região com atividade)
  2 — sunspot_group  (grupo de manchas visíveis)
"""

import numpy as np
import cv2
from pathlib import Path
import random
import yaml

DATASET_PATH = Path("/content/solar_synthetic")
IMG_SIZE = 640
N_TRAIN, N_VAL, N_TEST = 240, 40, 20
CLASSES = ["quiet_sun", "active_region", "sunspot_group"]

for split in ["train", "val", "test"]:
    (DATASET_PATH / "images" / split).mkdir(parents=True, exist_ok=True)
    (DATASET_PATH / "labels" / split).mkdir(parents=True, exist_ok=True)


def make_solar_image(n_spots: int, img_size: int = IMG_SIZE):
    """Gera uma imagem sintética de disco solar com manchas."""
    img = np.zeros((img_size, img_size, 3), dtype=np.uint8)
    cx, cy = img_size // 2, img_size // 2
    r = int(img_size * 0.45)

    # Disco solar — gradiente amarelo-laranja
    for i in range(img_size):
        for j in range(img_size):
            dist = np.sqrt((i - cy)**2 + (j - cx)**2)
            if dist <= r:
                factor = 1.0 - 0.3 * (dist / r)**2
                img[i, j] = [
                    int(255 * factor * 0.9),   # B
                    int(255 * factor * 0.85),  # G
                    int(255 * factor),          # R
                ]

    annotations = []
    for _ in range(n_spots):
        # Posição aleatória dentro do disco
        angle = random.uniform(0, 2 * np.pi)
        dist  = random.uniform(0, r * 0.8)
        sx = int(cx + dist * np.cos(angle))
        sy = int(cy + dist * np.sin(angle))

        # Tamanho da mancha
        sw = random.randint(15, 60)
        sh = random.randint(12, 50)

        # Determina classe pela intensidade da mancha
        darkness = random.uniform(0.05, 0.35)
        cls = 2 if darkness < 0.15 else 1  # sunspot_group ou active_region

        # Desenha penumbra (cinza) + umbra (escura)
        cv2.ellipse(img, (sx, sy), (sw, sh), 0, 0, 360, (80, 80, 80), -1)
        cv2.ellipse(img, (sx, sy), (sw//2, sh//2), 0, 0, 360,
                    (int(255*darkness), int(255*darkness), int(255*darkness)), -1)

        # Anotação YOLO: cx cy w h normalizados
        ann_cx = sx / img_size
        ann_cy = sy / img_size
        ann_w  = (sw * 2) / img_size
        ann_h  = (sh * 2) / img_size
        ann_cx = max(0, min(1, ann_cx))
        ann_cy = max(0, min(1, ann_cy))
        ann_w  = max(0.01, min(1, ann_w))
        ann_h  = max(0.01, min(1, ann_h))
        annotations.append(f"{cls} {ann_cx:.6f} {ann_cy:.6f} {ann_w:.6f} {ann_h:.6f}")

    return img, annotations


def generate_split(split: str, n: int):
    img_dir = DATASET_PATH / "images" / split
    lbl_dir = DATASET_PATH / "labels" / split
    for i in range(n):
        n_spots = random.choices([0, 1, 2, 3, 4], weights=[5, 20, 35, 25, 15])[0]
        img, anns = make_solar_image(n_spots)
        fname = f"solar_{split}_{i:04d}"
        cv2.imwrite(str(img_dir / f"{fname}.jpg"), img)
        with open(lbl_dir / f"{fname}.txt", "w") as f:
            f.write("\n".join(anns))
    print(f"  {split:5s}: {n} imagens geradas")


print("Gerando dataset sintético solar...")
generate_split("train", N_TRAIN)
generate_split("val",   N_VAL)
generate_split("test",  N_TEST)

# Salvar dataset.yaml
ds_yaml = {
    "path": str(DATASET_PATH),
    "train": "images/train",
    "val":   "images/val",
    "test":  "images/test",
    "nc": 3,
    "names": CLASSES,
}
with open(DATASET_PATH / "dataset.yaml", "w") as f:
    yaml.dump(ds_yaml, f, default_flow_style=False)

print(f"\n✅ Dataset sintético criado em: {DATASET_PATH}")
print(f"   Total: {N_TRAIN + N_VAL + N_TEST} imagens | Classes: {CLASSES}")

## Célula 4 — Treinar YOLOv8n (Fine-tuning)

> **Tempo estimado com T4 GPU:** ~8–12 minutos para 30 épocas.  
> Ajuste `DATASET_YAML` para o path correto dependendo de qual Célula 3 você executou.

In [ ]:
from ultralytics import YOLO
from pathlib import Path

# ─── Configurar caminho do dataset ───────────────────────────────────────────
# Se usou Célula 3 (Roboflow): use o path retornado por dataset.location + /data.yaml
# Se usou Célula 3B (Sintético): use o path abaixo
DATASET_YAML = "/content/solar_synthetic/dataset.yaml"
# DATASET_YAML = f"{dataset.location}/data.yaml"  # ← descomente se usou Roboflow
# ─────────────────────────────────────────────────────────────────────────────

OUTPUT_DIR = Path("/content/yolo_output")
OUTPUT_DIR.mkdir(exist_ok=True)

print("=" * 60)
print("  HeliOS YOLO — Fine-tuning YOLOv8n")
print(f"  Dataset : {DATASET_YAML}")
print(f"  Saída   : {OUTPUT_DIR}")
print("=" * 60)

# Carrega YOLOv8n pré-treinado no COCO (pesos públicos, ~6 MB)
model = YOLO("yolo11n.pt")

# Fine-tuning
results = model.train(
    data=DATASET_YAML,
    epochs=30,
    imgsz=640,
    batch=16,
    project=str(OUTPUT_DIR),
    name="helios_yolo",
    exist_ok=True,
    device=0,        # GPU 0
    verbose=True,
    patience=10,     # EarlyStopping se mAP não melhorar em 10 épocas
    save_period=10,  # salva checkpoint a cada 10 épocas
)

print("\n✅ Treinamento concluído!")
best = Path(results.save_dir) / "weights" / "best.pt"
print(f"   Melhores pesos: {best}")

## Célula 5 — Avaliar Métricas e Exportar Resultados

In [ ]:
import json, shutil
from pathlib import Path
from ultralytics import YOLO

OUTPUT_DIR = Path("/content/yolo_output")
SAVE_DIR   = OUTPUT_DIR / "helios_yolo"
BEST_PT    = SAVE_DIR / "weights" / "best.pt"

# Carrega melhores pesos
model = YOLO(str(BEST_PT))

# Avalia no conjunto de validação
print("Avaliando no conjunto de validação...")
metrics = model.val(verbose=False)

map50    = float(metrics.box.map50)
map5095  = float(metrics.box.map)
prec     = float(metrics.box.mp)
recall   = float(metrics.box.mr)

print(f"\n{'='*50}")
print(f"  HeliOS YOLO — Métricas de Validação")
print(f"{'='*50}")
print(f"  mAP@0.5       : {map50:.4f}  {'✅' if map50 > 0.5 else '⚠️ (abaixo do alvo 0.5)'}")
print(f"  mAP@0.5:0.95  : {map5095:.4f}")
print(f"  Precision     : {prec:.4f}")
print(f"  Recall        : {recall:.4f}")
print(f"{'='*50}")

# Salvar métricas em JSON
meta = {
    "map50":         map50,
    "map50_95":      map5095,
    "precision":     prec,
    "recall":        recall,
    "epochs_trained": 30,
    "model_base":    "yolo11n.pt",
    "classes":       ["quiet_sun", "active_region", "sunspot_group"],
    "trained_on":    "Google Colab T4 GPU",
}
with open(OUTPUT_DIR / "yolo_metrics.json", "w") as f:
    json.dump(meta, f, indent=2)

# Copiar best.pt para a saída raiz
shutil.copy(BEST_PT, OUTPUT_DIR / "helios_yolo.pt")

# Copiar gráfico de resultados
results_png = SAVE_DIR / "results.png"
if results_png.exists():
    shutil.copy(results_png, OUTPUT_DIR / "yolo_results.png")
    print(f"\n  Gráfico salvo em: {OUTPUT_DIR / 'yolo_results.png'}")

print(f"\n  Arquivos prontos para download em: {OUTPUT_DIR}")
print(f"  - helios_yolo.pt       → src/ml/yolo/model/")
print(f"  - yolo_metrics.json    → src/ml/yolo/model/")
print(f"  - yolo_results.png     → docs/")

## Célula 6 — Testar Inferência em Imagem SDO Real

In [ ]:
import urllib.request
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from ultralytics import YOLO
from pathlib import Path

OUTPUT_DIR = Path("/content/yolo_output")
BEST_PT    = OUTPUT_DIR / "helios_yolo.pt"

# Baixar imagem SDO/AIA 193 mais recente (canal EUV — mostra regiões ativas em brilho)
SDO_URL  = "https://sdo.gsfc.nasa.gov/assets/img/latest/latest_1024_0193.jpg"
SDO_PATH = Path("/content/sdo_latest_0193.jpg")
print("Baixando imagem SDO/AIA 193 Å mais recente...")
urllib.request.urlretrieve(SDO_URL, SDO_PATH)
print(f"  Salva em: {SDO_PATH}")

# Inferência
model = YOLO(str(BEST_PT))
results = model.predict(str(SDO_PATH), conf=0.20, verbose=False)
result  = results[0]

# Salvar imagem anotada
annotated_path = OUTPUT_DIR / "sdo_annotated.jpg"
result.save(str(annotated_path))

# Mostrar resultado
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(mpimg.imread(str(SDO_PATH)))
axes[0].set_title("Imagem SDO/AIA 193 Å — Original", fontsize=11)
axes[0].axis("off")

axes[1].imshow(mpimg.imread(str(annotated_path)))
axes[1].set_title(f"Detecções HeliOS YOLO — {len(result.boxes)} objetos", fontsize=11)
axes[1].axis("off")

plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / "yolo_inference_demo.png"), dpi=150, bbox_inches="tight")
plt.show()

# Sumário das detecções
print(f"\nDetecções ({len(result.boxes)} total):")
for box in result.boxes:
    cls_name = model.names[int(box.cls[0])]
    conf     = float(box.conf[0])
    print(f"  [{cls_name}]  conf={conf:.2f}")

# Copiar para output
# import shutil
# shutil.copy(OUTPUT_DIR / "yolo_inference_demo.png", OUTPUT_DIR / "yolo_inference_demo.png")

## Célula 7 — Download dos Artefatos

Baixa todos os arquivos necessários para colocar no repositório local.

In [ ]:
from google.colab import files
from pathlib import Path

OUTPUT_DIR = Path("/content/yolo_output")

download_files = {
    "helios_yolo.pt":           "→ src/ml/yolo/model/helios_yolo.pt",
    "yolo_metrics.json":        "→ src/ml/yolo/model/yolo_metrics.json",
    "yolo_results.png":         "→ docs/yolo_results.png",
    "yolo_inference_demo.png":  "→ docs/yolo_inference_demo.png",
}

print("Baixando artefatos...\n")
for fname, dest in download_files.items():
    fpath = OUTPUT_DIR / fname
    if fpath.exists():
        print(f"  ✅ {fname}  {dest}")
        files.download(str(fpath))
    else:
        print(f"  ⚠️  {fname} não encontrado — verifique se as células anteriores foram executadas")

print("\nInstruções:")
print("  1. helios_yolo.pt      → copie para: src/ml/yolo/model/")
print("  2. yolo_metrics.json   → copie para: src/ml/yolo/model/")
print("  3. yolo_results.png    → copie para: docs/")
print("  4. yolo_inference_demo.png → copie para: docs/")